# Chương 7: Xây dựng Ứng dụng Trò chuyện
## Bắt đầu nhanh với API OpenAI

Sổ tay này được điều chỉnh từ [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) bao gồm các sổ tay truy cập dịch vụ [Azure OpenAI](notebook-azure-openai.ipynb).

API OpenAI Python cũng hoạt động với các Mô hình Azure OpenAI, với một vài chỉnh sửa. Tìm hiểu thêm về sự khác biệt tại đây: [Cách chuyển đổi giữa các endpoints OpenAI và Azure OpenAI với Python](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Tổng quan  
"Các mô hình ngôn ngữ lớn là các hàm ánh xạ từ văn bản sang văn bản. Với một chuỗi văn bản đầu vào, một mô hình ngôn ngữ lớn cố gắng dự đoán văn bản sẽ xuất hiện tiếp theo"(1). Sổ tay "khởi động nhanh" này sẽ giới thiệu cho người dùng các khái niệm cấp cao về LLM, các yêu cầu gói cốt lõi để bắt đầu với AML, một giới thiệu nhẹ về thiết kế prompt, và một số ví dụ ngắn về các trường hợp sử dụng khác nhau. 


## Mục lục  

[Tổng quan](#overview)  
[Cách sử dụng Dịch vụ OpenAI](#how-to-use-openai-service)  
[1. Tạo Dịch vụ OpenAI của bạn](#1.-creating-your-openai-service)  
[2. Cài đặt](#2.-installation)    
[3. Thông tin đăng nhập](#3.-credentials)  

[Các trường hợp sử dụng](#use-cases)    
[1. Tóm tắt văn bản](#1.-summarize-text)  
[2. Phân loại văn bản](#2.-classify-text)  
[3. Tạo tên sản phẩm mới](#3.-generate-new-product-names)  
[4. Tinh chỉnh bộ phân loại](#4.fine-tune-a-classifier)  

[Tài liệu tham khảo](#references)


### Xây dựng prompt đầu tiên của bạn  
Bài tập ngắn này sẽ cung cấp một giới thiệu cơ bản về cách gửi prompt tới một mô hình OpenAI cho một tác vụ đơn giản "tóm tắt".


**Các bước**:  
1. Cài đặt thư viện OpenAI trong môi trường Python của bạn  
2. Tải các thư viện trợ giúp chuẩn và thiết lập thông tin bảo mật OpenAI điển hình cho Dịch vụ OpenAI mà bạn đã tạo  
3. Chọn một mô hình cho tác vụ của bạn  
4. Tạo một prompt đơn giản cho mô hình  
5. Gửi yêu cầu của bạn đến API của mô hình!


### 1. Cài đặt OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Nhập thư viện trợ giúp và khởi tạo thông tin xác thực


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Tìm mô hình phù hợp  
Các mô hình như GPT-4o và GPT-4o mini có thể hiểu và tạo ra ngôn ngữ tự nhiên, và có sẵn trên nền tảng OpenAI với các mức độ mạnh mẽ và tốc độ khác nhau phù hợp cho các nhiệm vụ khác nhau.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. Thiết Kế Prompt  

"Phép màu của các mô hình ngôn ngữ lớn là bằng cách được đào tạo để giảm thiểu sai số dự đoán này qua một lượng lớn văn bản, các mô hình cuối cùng học được những khái niệm hữu ích cho các dự đoán đó. Ví dụ, chúng học các khái niệm như"(1):

* cách đánh vần
* cách ngữ pháp hoạt động
* cách diễn giải lại
* cách trả lời câu hỏi
* cách giữ một cuộc trò chuyện
* cách viết bằng nhiều ngôn ngữ
* cách viết mã
* v.v.

#### Cách kiểm soát một mô hình ngôn ngữ lớn  
"Trong tất cả các đầu vào tới một mô hình ngôn ngữ lớn, phần có ảnh hưởng nhất là prompt văn bản(1).

Các mô hình ngôn ngữ lớn có thể được gợi ý để tạo ra đầu ra theo vài cách:

Hướng dẫn: Nói cho mô hình biết bạn muốn gì
Hoàn thành: Kích thích mô hình hoàn thành phần bắt đầu của điều bạn muốn
Minh họa: Cho mô hình thấy bạn muốn gì, bằng cách:
Một vài ví dụ trong prompt
Nhiều trăm hoặc nghìn ví dụ trong tập dữ liệu huấn luyện tinh chỉnh"



#### Có ba hướng dẫn cơ bản để tạo prompt:

**Hiển thị và nói rõ**. Làm rõ bạn muốn gì bằng hướng dẫn, ví dụ, hoặc kết hợp cả hai. Nếu bạn muốn mô hình xếp hạng một danh sách các mục theo thứ tự chữ cái hoặc phân loại một đoạn văn theo cảm xúc, hãy cho nó thấy đó là điều bạn muốn.

**Cung cấp dữ liệu chất lượng**. Nếu bạn đang cố xây dựng bộ phân loại hoặc khiến mô hình tuân theo một mô hình, hãy đảm bảo có đủ ví dụ. Hãy chắc chắn kiểm tra kỹ các ví dụ của bạn — mô hình thường đủ thông minh để nhận ra các lỗi đánh vần cơ bản và đưa ra phản hồi, nhưng nó cũng có thể nghĩ đó là cố ý và điều đó có thể ảnh hưởng đến phản hồi.

**Kiểm tra cài đặt của bạn.** Thiết lập nhiệt độ (temperature) và top_p kiểm soát mức độ quyết đoán của mô hình khi tạo phản hồi. Nếu bạn đang yêu cầu một phản hồi chỉ có một câu trả lời đúng duy nhất, bạn nên đặt các giá trị này thấp hơn. Nếu bạn muốn có phản hồi đa dạng hơn, thì có thể đặt chúng cao hơn. Sai lầm phổ biến nhất người ta mắc phải với các thiết lập này là nghĩ chúng là điều khiển "sự thông minh" hay "sáng tạo".


Nguồn: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Gửi đi! 


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Lặp lại cùng một cuộc gọi, kết quả so sánh thế nào?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Tóm Tắt Văn Bản  
#### Thử Thách  
Tóm tắt văn bản bằng cách thêm 'tl;dr:' vào cuối đoạn văn bản. Lưu ý cách mô hình hiểu và thực hiện nhiều nhiệm vụ khác nhau mà không cần hướng dẫn thêm. Bạn có thể thử nghiệm với các lời nhắc mô tả hơn so với tl;dr để điều chỉnh hành vi của mô hình và tùy chỉnh bản tóm tắt bạn nhận được(3).  

Các nghiên cứu gần đây đã chứng minh sự cải thiện đáng kể trên nhiều nhiệm vụ và bộ chuẩn NLP bằng cách tiền huấn luyện trên một kho văn bản lớn sau đó tinh chỉnh trên một nhiệm vụ cụ thể. Mặc dù thường không phụ thuộc nhiệm vụ trong kiến trúc, phương pháp này vẫn đòi hỏi bộ dữ liệu tinh chỉnh đặc thù cho nhiệm vụ gồm hàng nghìn hoặc hàng chục nghìn ví dụ. Ngược lại, con người thường có thể thực hiện một nhiệm vụ ngôn ngữ mới chỉ từ vài ví dụ hoặc từ hướng dẫn đơn giản - điều mà các hệ thống NLP hiện tại vẫn còn gặp nhiều khó khăn. Ở đây chúng tôi cho thấy việc mở rộng mô hình ngôn ngữ cải thiện đáng kể hiệu suất không phụ thuộc nhiệm vụ với vài ví dụ, thậm chí đôi khi đạt được hiệu quả cạnh tranh với các phương pháp tinh chỉnh tiên tiến trước đây.  



Tl;dr


# Bài tập cho một số trường hợp sử dụng  
1. Tóm tắt văn bản  
2. Phân loại văn bản  
3. Tạo tên sản phẩm mới


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Phân loại văn bản  
#### Thử thách  
Phân loại các mục vào các danh mục được cung cấp vào thời điểm suy luận. Trong ví dụ sau, chúng tôi cung cấp cả các danh mục và văn bản để phân loại trong lời nhắc(*playground_reference). 

Yêu cầu khách hàng: Xin chào, một trong các phím trên bàn phím laptop của tôi gần đây bị hỏng và tôi cần thay thế:

Danh mục phân loại:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Tạo Tên Sản Phẩm Mới
#### Thử Thách
Tạo tên sản phẩm từ các từ ví dụ. Ở đây, chúng tôi đưa vào lời nhắc thông tin về sản phẩm mà chúng tôi sẽ tạo tên cho nó. Chúng tôi cũng cung cấp một ví dụ tương tự để thể hiện mẫu mà chúng tôi muốn nhận được. Chúng tôi cũng đã đặt giá trị nhiệt độ cao để tăng tính ngẫu nhiên và phản hồi sáng tạo hơn.

Mô tả sản phẩm: Máy làm sữa lắc tại nhà
Từ khóa gốc: nhanh, khỏe mạnh, nhỏ gọn.
Tên sản phẩm: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Mô tả sản phẩm: Một đôi giày có thể phù hợp với mọi kích cỡ bàn chân.
Từ khóa gốc: thích nghi, vừa vặn, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Tài liệu tham khảo  
- [Sổ tay Openai](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Cổng Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Các phương pháp tốt nhất để tinh chỉnh các mô hình GPT phân loại văn bản](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Để được hỗ trợ thêm  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Người đóng góp
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
